# 🔬 GIADA Task 9c — replay NEURON denso
Dodici transizioni **train** già congelate nella Task 9. Nessun nuovo protocollo, training o test sigillato. Il replay a 40 campioni deve coincidere prima di interpretare quelli più densi.

In [ ]:
from pathlib import Path
import base64, json, os, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_task_9c');GIADA_REPO=WORK/'giada';TEACHER_REPO=WORK/'neuron_as_deep_net'
assert not GIADA_REPO.exists() and not TEACHER_REPO.exists(),'Sessione già inizializzata: usa una sessione Kaggle nuova.'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip();print({'revision':REVISION})


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7','numpy','pandas','matplotlib','h5py','pyarrow','pyyaml'],check=True)
SIMULATION_DIR=TEACHER_REPO/'L5PC_NEURON_simulation'
if not list(SIMULATION_DIR.rglob('libnrnmech.so')):
 nrnivmodl=shutil.which('nrnivmodl') or str(Path(sys.executable).parent/'nrnivmodl')
 subprocess.run([nrnivmodl,'mods'],cwd=SIMULATION_DIR,check=True)
assert list(SIMULATION_DIR.rglob('libnrnmech.so')),'Compilazione MOD fallita'
print('NEURON e meccanismi canonici pronti')


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]:del sys.modules[name]
from src.giada_teacher import ExtractedGateFormula, evaluate_dense_replay
from src.giada_teacher.physiological_path_floor import load_verified_task9
plan=json.loads((GIADA_REPO/'experiments/teacher_dense_physiological_path_replay_plan_v1.json').read_text())
display({'task':plan['task'],'paths':plan['path_count'],'intervals_ms':plan['sample_intervals_ms'],'gpu_required':False})


## 📁 Input
Servono il **dataset targeted v1.1 base completo, inclusi gli snapshot nativi**, e l'artefatto `giada_physiological_voltage_paths.zip` della Task 9. Non occorre ricaricare il piccolo ZIP della Task 9b: il suo esito è già registrato nel piano.

In [ ]:
INPUT_ROOT=Path('/kaggle/input');override=os.environ.get('GIADA_TASK9_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT_ROOT.is_dir():
 candidates+=list(INPUT_ROOT.rglob('giada_physiological_voltage_paths.zip'))
 candidates += [p for p in INPUT_ROOT.rglob('archive.zip') if 'physiological' in str(p).lower()]
 candidates += [p.parent for p in INPUT_ROOT.rglob('selected_paths.json') if (p.parent/'final_report.json').is_file()]
TASK9_SOURCE=None
for path in candidates:
 if not path.exists():continue
 try:load_verified_task9(path);TASK9_SOURCE=path.resolve();break
 except (RuntimeError,FileNotFoundError,ValueError,KeyError):continue
assert TASK9_SOURCE is not None,'Artefatto Task 9 esatto non trovato.'
base_override=os.environ.get('GIADA_TARGETED_DATASET')
base_candidates=[Path(base_override).expanduser()] if base_override else []
base_candidates += [Path('/kaggle/input/datasets/alessandrobelli/hayflow-targeted-transition-dataset-v1-1-base')]
if INPUT_ROOT.is_dir():base_candidates += [p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower()]
BASE_ROOT=next((p.resolve() for p in base_candidates if p.is_dir() and all((p/name).is_file() for name in ('transition_dataset.h5','state_schema.json','dataset_manifest.json')) and (p/'snapshots').is_dir()),None)
assert BASE_ROOT is not None,'Dataset targeted v1.1 base completo non trovato; servono HDF5, manifest, schema e snapshots nella stessa cartella.'
print({'task9_source':str(TASK9_SOURCE),'base_root':str(BASE_ROOT)})


In [ ]:
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_dense_physiological_teacher_replay')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Usa una sessione nuova.'
def progress(percent,label):print(f'[GIADA Task 9c][SHA-256 {label}] {percent}%',flush=True)
report=evaluate_dense_replay(formula,BASE_ROOT,TASK9_SOURCE,TEACHER_REPO,GIADA_REPO,OUTPUT_DIR,code_revision=REVISION,progress=progress)
display({'valid':report['valid'],'paths':report.get('path_count'),'blocker':report.get('blocker'),'dense_floor_calibrated':report.get('dense_teacher_formula_floor_calibrated'),'dense_convergence':report.get('dense_formula_convergence_max_abs'),'methods':report.get('metrics')})
if not report['valid']:print('Replay canonico fallito: niente attribuzione causale. Scarica comunque il report di blocco.')


In [ ]:
import pandas as pd
rows=[]
for key,group in report.get('groups',{}).items():
 rows.append({'site_regime':key,'n':group['count'],'40_sample_m':group['methods']['0.025']['m_rmse'],'interpolated_m':group['methods']['linear_20']['m_rmse'],'dense_0.005_m':group['methods']['0.005']['m_rmse'],'dense_0.001_m':group['methods']['0.001']['m_rmse']})
if rows:display(pd.DataFrame(rows))
if report['valid'] and not report['dense_teacher_formula_floor_calibrated']:print('Il floor denso resta non calibrato: non attribuire la differenza residua alla LUT/rete.')


## 📦 Download ZIP
L'archivio include solo il report piccolo, non gli snapshot o l'HDF5.

In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_dense_physiological_teacher_replay','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
